# LIGO Glitch Classifier

Convolutional-network classifier for transient noise artifacts ("glitches")
in LIGO gravitational-wave detector data, benchmarked against the
[Gravity Spy](https://www.zooniverse.org/projects/zooniverse/gravity-spy)
citizen-science labeling project.

**Background.** LIGO detects gravitational waves by measuring sub-atomic
length changes in its interferometer arms, which makes the instrument
extremely susceptible to non-astrophysical noise transients from seismic
activity, scattered light, electronics, and mechanical resonances. These
glitches can mimic or mask real astrophysical signals, so classifying them
is a standard part of LIGO detector characterization. Gravity Spy is the
reference dataset and labeling scheme for this task: it represents each
glitch as a set of time-frequency spectrogram images (Q-transforms) at four
duration windows (0.5 s, 1 s, 2 s, 4 s), labeled by a combination of
citizen-science volunteers and a production ML classifier.

**Approach.** A ResNet18 backbone, fine-tuned from ImageNet weights, is
trained on the 1.0 s duration-view spectrograms from the Gravity Spy
training set. Two runs are reported: a baseline on 8 well-separated classes,
and a full run on the complete 22-class taxonomy. Predictions are then
benchmarked against Gravity Spy's own published ML labels and its volunteer
consensus labels for the same glitches.

**Repository:** https://github.com/amishi71/ligo-glitch-classifier

**Runtime:** requires a GPU (Runtime → Change runtime type → T4 GPU).

## 1. Environment setup

Clone the repository and install dependencies.

In [ ]:
REPO_URL = "https://github.com/amishi71/ligo-glitch-classifier.git"

!git clone $REPO_URL
%cd ligo-glitch-classifier
!pwd

In [ ]:
!pip install -q -r requirements.txt

## 2. Data acquisition

Downloads the Gravity Spy training set (`trainingsetv1d1.h5`, ~2.8 GB) from
Zenodo. This file contains pre-rendered spectrogram images for every
labeled glitch, organized by class and by train/validation/test split.

Label CSVs for benchmarking (Section 6) are downloaded separately later,
since they are only needed after a model has been trained.

In [ ]:
!python src/download_data.py --training-set

## 3. Dataset structure check

The HDF5 file's internal split names (`train` / `validation` / `test`) are
verified explicitly before training, since `dataset.py` requires an exact
string match against whatever split names are actually stored in the file.
A mismatch here fails silently — it produces an empty dataset rather than
an error — so this check is run as a first step rather than discovered
during training.

In [ ]:
import h5py

with h5py.File("data/raw/trainingsetv1d1.h5", "r") as f:
    class_names = list(f.keys())
    # any class works as a probe since split names are consistent across classes
    probe_class = "Blip" if "Blip" in class_names else class_names[0]
    split_names = list(f[probe_class].keys())

print(f"Classes ({len(class_names)}): {class_names}")
print(f"Splits: {split_names}")

# dataset.py defaults to ['train', 'validation', 'test']; if the file uses
# different names, override with --split-train/--split-val/--split-test
# in the train.py and evaluate.py calls below.
assert split_names, "No splits found under the probe class — check the HDF5 file."

## 4. Baseline: 8-class training

Trains on a subset of 8 well-separated glitch classes (`Blip`, `Chirp`,
`Koi_Fish`, `Low_Frequency_Burst`, `Power_Line`, `Scattered_Light`,
`Violin_Mode`, `Whistle`) as a fast sanity check before committing to the
full 22-class run. ResNet18, ImageNet-pretrained, 15 epochs, class-weighted
cross-entropy loss to account for per-class sample count differences.
Checkpoints are saved to `checkpoints/` whenever validation accuracy
improves.

In [ ]:
!python src/train.py --epochs 15

## 5. Baseline evaluation

Test-set accuracy, macro-averaged F1, and a per-class precision/recall/F1 report.

In [ ]:
!python src/evaluate.py --checkpoint checkpoints/best_model.pt

Confusion matrix for the 8-class baseline:

In [ ]:
from IPython.display import Image, display

display(Image("outputs/confusion_matrix.png"))

## 6. Checkpoint backup

Colab sessions are ephemeral — the filesystem is wiped on disconnect. Model
checkpoints and evaluation outputs are copied to Google Drive to persist
them across sessions.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os
import shutil

backup_dir = "/content/drive/MyDrive/ligo_glitch_backup"
os.makedirs(backup_dir, exist_ok=True)

for folder in ["checkpoints", "outputs"]:
    dest = os.path.join(backup_dir, folder)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(folder, dest)
    print(f"Backed up {folder}/ -> {dest}")

## 7. Full training: all 22 classes

The 8-class baseline confirms the pipeline works, but is not directly
comparable to Gravity Spy's published results, which cover the complete
taxonomy. This run trains on all 22 classes and saves to a separate
checkpoint directory so the baseline model is preserved.

In [ ]:
!python src/train.py --all-classes --epochs 15 --checkpoint-dir checkpoints_22class

## 8. Full-model evaluation

In [ ]:
!python src/evaluate.py --checkpoint checkpoints_22class/best_model.pt --output-dir outputs_22class

In [ ]:
from IPython.display import Image, display

display(Image("outputs_22class/confusion_matrix.png"))

## 9. Gravity Spy label download

Downloads the official Gravity Spy ML-classification files and the
volunteer+ML consensus labels for benchmarking (Section 10).

The ML label files are split by detector and observing run, and the
training-set HDF5 doesn't record which run each glitch came from, so
`--ml-labels all` downloads all eight (`H1_O1`, `H1_O2`, `H1_O3a`, `H1_O3b`,
`L1_O1`, `L1_O2`, `L1_O3a`, `L1_O3b`) to guarantee coverage rather than
guessing. Section 10 prints the match rate against whatever files were
downloaded, as a diagnostic.

The volunteer-labels file is a single pre-aggregated HDF5
(`retired_fulldata_min2_max50_ret0p9.hdf5`) with one row per glitch,
`gravityspy_id`, and a `final_label` (the combined volunteer + ML
consensus) already computed.

In [ ]:
!python src/download_data.py --ml-labels all --volunteer-labels

## 10. Benchmark against Gravity Spy labels

Joins the trained model's test-set predictions to the official Gravity Spy
labels on `gravityspy_id` (the shared identifier across the training-set
HDF5, the ML label CSVs, and the volunteer+ML consensus HDF5) and reports:

- Agreement between the model's predictions and Gravity Spy's own ML
  classifier's labels
- Agreement between the model's predictions and the volunteer+ML
  consensus labels (`final_label`)
- A reference point: agreement between Gravity Spy's ML label and the
  training-set ground-truth label

A full per-glitch comparison table is written to
`outputs_22class/benchmark_results.csv` for error analysis — in particular,
cases where the model disagrees with the volunteer label are often
genuinely ambiguous glitches rather than model errors, and are worth
inspecting individually.

In [ ]:
!python src/benchmark.py \
    --checkpoint checkpoints_22class/best_model.pt \
    --ml-labels data/labels/H1_O1.csv data/labels/H1_O2.csv data/labels/H1_O3a.csv data/labels/H1_O3b.csv \
                data/labels/L1_O1.csv data/labels/L1_O2.csv data/labels/L1_O3a.csv data/labels/L1_O3b.csv \
    --volunteer-labels data/labels/retired_fulldata_min2_max50_ret0p9.hdf5 \
    --output-dir outputs_22class

## 11. Final backup

Persists the 22-class checkpoint, evaluation outputs, and benchmark results
to Google Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os
import shutil

backup_dir = "/content/drive/MyDrive/ligo_glitch_backup"
os.makedirs(backup_dir, exist_ok=True)

for folder in ["checkpoints_22class", "outputs_22class", "src/benchmark.py"]:
    dest = os.path.join(backup_dir, os.path.basename(folder))
    if os.path.isdir(folder):
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
    else:
        shutil.copy(folder, dest)
    print(f"Backed up {folder} -> {dest}")